### Daten erstellen und customclassifier fitten

In [1]:
from utils import create_train_test_data, create_new_dataset_with_ipcw_weights
from class_DecisionTreeBaggingClassifier import DecisionTreeBaggingClassifier
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter

n = 20
seed = 42
tau = 32
B_RF = int(0.7 * n * 2)
B_first_level = 20

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': seed, 'tau': tau}


df_train, df_test\
    , n_events_after_cut_train, portion_censored_after_cut_train\
    , n_events_after_cut_test, portion_censored_after_cut_test = create_train_test_data(params=params_data_creation)

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  seed,
                'weighted_bootstrapping':     True,  }

X_pred_point = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})
clf = DecisionTreeBaggingClassifier(params_rf)

clf.fit(
            X=df_train.drop(
                ["time", "event", "weights_ipcw", "survived"], axis=1
            ).values,
            y=df_train["survived"].values,
            sample_weights=df_train["weights_ipcw"].values,)



In [3]:

# Precompute reusable values and objects
n = df_train.shape[0]
rng = np.random.default_rng(seed)

# Generate bootstrapped indices once
first_level_boot_indices = rng.choice(
    np.arange(n), size=(B_first_level, n), replace=True
)

# Convert the entire DataFrame to a Numpy array to avoid repeated use of .iloc
df_train_np = df_train.values
columns_to_drop = ["time", "event", "weights_ipcw", "survived"]

# Identify the column indices for 'time', 'event', and other relevant columns
time_col_idx = df_train.columns.get_loc("time")
event_col_idx = df_train.columns.get_loc("event")
columns_to_drop_indices = [df_train.columns.get_loc(col) for col in columns_to_drop]

# Preallocate the classifier and Kaplan-Meier fitter objects
clf = DecisionTreeBaggingClassifier(params_rf)
kmf = KaplanMeierFitter()

preds = np.empty(B_first_level)

for b in range(B_first_level):
    # Direct Numpy indexing instead of .iloc
    boot_indices = first_level_boot_indices[b]
    df_train_boot = df_train_np[boot_indices, :]
    clf = DecisionTreeBaggingClassifier(params_rf)
    # Extract the relevant columns using the identified indices
    time_values = df_train_boot[:, time_col_idx]
    event_values = df_train_boot[:, event_col_idx]

    # Fit the Kaplan-Meier estimator using Numpy arrays
    kmf.fit(
        durations = np.array(time_values, dtype=float),
        event_observed=np.array((1 - event_values),dtype=bool),
    )

    # Create the new dataset with IPCW weights using the optimized function
    df_train_ipcw = create_new_dataset_with_ipcw_weights(
        data=pd.DataFrame(df_train_boot, columns=df_train.columns), t=tau
    )

    # Prepare the data for fitting by dropping the unnecessary columns using numpy
    X_train_ipcw = np.delete(df_train_ipcw.values, columns_to_drop_indices, axis=1)
    y_train_ipcw = df_train_ipcw["survived"].values
    weights_ipcw = df_train_ipcw["weights_ipcw"].values

    # Set the random state and fit the classifier
    clf.set_random_state(random_state=seed + b)
    clf.fit(X=X_train_ipcw, y=y_train_ipcw, sample_weights=weights_ipcw)
    # Predict and store result
    _ ,x = clf.predict_proba(X_pred_point.values)



Updated random_state to: 42
0
Updated random_state to: 43
1
Updated random_state to: 44
2
Updated random_state to: 45
3
Updated random_state to: 46
4
Updated random_state to: 47
5
Updated random_state to: 48
6
Updated random_state to: 49
7
Updated random_state to: 50
8
Updated random_state to: 51
9
Updated random_state to: 52
10
Updated random_state to: 53
11
Updated random_state to: 54
12
Updated random_state to: 55
13
Updated random_state to: 56
14
Updated random_state to: 57
15
Updated random_state to: 58
16
Updated random_state to: 59
17
Updated random_state to: 60
18
Updated random_state to: 61
19


In [18]:
print(np.array((1 - event_values),dtype=bool))
print(np.array(time_values, dtype=float))

[ True  True  True False False  True  True  True  True  True  True  True
  True  True]
[ 91.50851501 272.9310447  164.12714096  37.3296256   37.3296256
 543.1589672   91.50851501 164.12714096  98.38213464  91.50851501
  28.62429886   4.5841417  272.9310447  272.9310447 ]
